In [101]:
from pymycobot import MyPalletizer260  
from pymycobot import MyPalletizerSocket
import numpy as np

In [102]:
# Puerto De conexion con el brazo
mc = MyPalletizer260('COM3')

In [103]:
speed = 50
angles = [0,0,0,0]
mc.send_angles(angles,speed)

## Envolvente J2 → J3: cómo cambian los límites de J3 en función de J2

El paralelogramo del MyPalletizer260 hace que el rango alcanzable de **J3 dependa de J2**. Lo único que el repo conoce hoy son cajas fijas: API J2 −2..90°, J3 −92..60°; XML `link1` 0..90°, `link2` 0..60.2°. Ninguna de las dos es la curva real.

**Los motores no se fuerzan nunca.** Nada de lo que sigue manda un comando de movimiento (`send_angle(s)`, `send_radians`, jog, `set_encoder(s)`). Los servos se **liberan** y el brazo se mueve **a mano**; por el puerto solo viajan `get_angles()`, `release_all_servos()` y `focus_servo()`.

**No corras la celda de `send_angles` de arriba** durante la medición. Solo hace falta la celda que abre el puerto (`mc`).

### Protocolo
1. Poné `MEDIR = True` y corré la celda de grabación. **El brazo queda flojo: sostenelo antes de apretar Enter.**
2. Empezá a mover apenas aprietes Enter: en los primeros 5 s tiene que verse movimiento en J2/J3, o la grabación se aborta (así se detecta un firmware que no reporta ángulos con los servos liberados, cosa que **no está verificada**).
3. Llevá J3 **suavemente** a su tope inferior y, manteniéndolo ahí, barré J2 lento de su mínimo a su máximo.
4. Llevá J3 a su tope superior y barré J2 de vuelta.
5. Repetí 3 y 4 una vez más. Un extremo solo cuenta como límite si de verdad se tocó el tope, así que el protocolo recorre el borde a propósito.
6. Al terminar (o con *Interrupt kernel*), **sostené el brazo** y apretá Enter: se reactivan los 4 servos. No está verificado que `focus_servo` mantenga la pose sin saltar.

Cada grabación se guarda en `data/raw/j2j3_envelope_raw*.csv` (grados API crudos) sin pisar la anterior. Conviene grabar dos veces: la diferencia entre las dos da el margen de seguridad.

Antes de medir, confirmá el cero de J2: la curva solo vale en los grados de la propia API.

In [105]:
# --- Envolvente J2 -> J3: grabacion a mano y ajuste ---------------------------
# NINGUNA funcion de esta celda manda un comando de movimiento. Lo unico que
# viaja por el puerto es get_angles(), release_all_servos() y focus_servo().
import time
from pathlib import Path

import numpy as np


def _leer(mc):
    """(4,) grados API, o None si la respuesta no vino o vino corta.

    get_angles() devuelve -1 ante un error, por eso el tipo se mira antes que
    el largo (un int no tiene len). Misma regla que MyPalletizerArm.read.
    """
    a = mc.get_angles()
    if not isinstance(a, (list, tuple)) or len(a) < 3:
        return None
    q = np.full(4, np.nan)
    q[: min(4, len(a))] = a[:4]
    return q


def _bins_vacios(j2, bin_deg, rango=(-2.0, 90.0)):
    """Centros de los bins de J2 (en `rango`, el limite API) sin ninguna muestra."""
    bordes = np.arange(rango[0], rango[1] + bin_deg, bin_deg)
    n, _ = np.histogram(j2, bins=bordes)
    return [f"{0.5 * (a + b):.1f}" for a, b, k in zip(bordes[:-1], bordes[1:], n) if k == 0]


def grabar_envolvente(mc, duracion_s=120.0, rate_hz=15.0, csv_path=None, *,
                      chequeo_s=5.0, mov_min_deg=2.0, bin_deg=5.0,
                      confirmar=input, reloj=time.perf_counter, dormir=time.sleep):
    """Graba (t, J1..J4) en grados API mientras el brazo se mueve A MANO.

    Libera los servos, lee get_angles() a `rate_hz` durante `duracion_s` (o
    hasta Ctrl-C / interrumpir el kernel) y los vuelve a activar SIEMPRE, aun
    si algo fallo. No manda ninguna consigna: los topes que se miden son los
    mecanicos, alcanzados por la mano del operador, nunca por un motor.

    Chequeo en los primeros `chequeo_s`: tienen que llegar lecturas y J2 o J3
    tienen que moverse al menos `mov_min_deg`. NO esta verificado que este
    firmware reporte angulos con los servos liberados; si no lo hace, esto
    levanta un error en vez de grabar una envolvente vacia que parezca buena.

    `confirmar`, `reloj` y `dormir` se inyectan para poder correrla contra un
    brazo de mentira sin esperar ni teclear.

    Devuelve un array (N, 5): t [s, perf_counter absoluto], J1, J2, J3, J4 [grados API].
    """
    q0 = _leer(mc)
    if q0 is None:
        raise RuntimeError("get_angles() no devolvio angulos; no se libera nada.")
    print("Pose actual (grados API): " + ", ".join(f"J{k + 1}={v:.1f}" for k, v in enumerate(q0)))
    confirmar(
        "Los servos se van a LIBERAR y el brazo queda flojo. Sostenelo, y "
        "empeza a mover J2/J3 apenas aprietes Enter... "
    )

    filas, fallas = [], 0
    mc.release_all_servos()
    try:
        periodo = 1.0 / rate_hz
        t0 = reloj()
        t_reporte = t0
        chequeado = False
        i = 0
        while True:
            ahora = reloj()
            if ahora - t0 >= duracion_s:
                break
            # Plazos absolutos (t0 + i/rate), no sleep(periodo) acumulado. Si
            # una lectura se atraso mas de un periodo, se resincroniza en vez
            # de correr a rafagas para "alcanzar".
            deadline = t0 + i * periodo
            if deadline > ahora:
                dormir(deadline - ahora)
            elif ahora - deadline > periodo:
                i = int((ahora - t0) / periodo)
            i += 1

            q = _leer(mc)
            t = reloj()
            if q is None:
                fallas += 1
            else:
                filas.append((t, *q))

            if not chequeado and t - t0 >= chequeo_s:
                chequeado = True
                esperadas = chequeo_s * rate_hz
                if len(filas) < 0.5 * esperadas:
                    raise RuntimeError(
                        f"Con los servos liberados llegaron {len(filas)} lecturas de "
                        f"~{esperadas:.0f} ({fallas} fallidas): el firmware no parece "
                        "reportar angulos asi. Nada grabado."
                    )
                d = np.array(filas)
                mov = np.ptp(d[:, 2:4], axis=0).max()
                if mov < mov_min_deg:
                    raise RuntimeError(
                        f"En {chequeo_s:.0f} s J2/J3 se movieron {mov:.2f} grados: o no "
                        "se movio el brazo, o la lectura no sigue al brazo liberado."
                    )
                print(f"Lectura OK: {len(filas)} muestras, J2/J3 se movieron {mov:.1f} grados.")

            if t - t_reporte >= 5.0 and filas:
                t_reporte = t
                j2 = np.array(filas)[:, 2]
                print(f"t={t - t0:5.0f} s  J2={filas[-1][2]:6.1f}  J3={filas[-1][3]:6.1f}  "
                      f"n={len(filas)}  bins de J2 sin muestras: {_bins_vacios(j2, bin_deg)}")
    except KeyboardInterrupt:
        print("Interrumpido: se guarda lo grabado hasta aca.")
    finally:
        confirmar("SOSTENE el brazo: Enter para reactivar los servos... ")
        for sid in (1, 2, 3, 4):
            mc.focus_servo(sid)

    datos = np.array(filas, dtype=np.float64).reshape(-1, 5)
    print(f"{len(datos)} muestras, {fallas} lecturas fallidas.")
    if csv_path is not None and len(datos):
        csv_path = Path(csv_path)
        if csv_path.exists():   # no pisar una grabacion anterior: se comparan
            csv_path = csv_path.with_name(f"{csv_path.stem}_{time.strftime('%Y%m%d_%H%M%S')}.csv")
        np.savetxt(csv_path, datos, delimiter=",", header="t,j1,j2,j3,j4",
                   comments="", fmt="%.6f")
        print(f"Guardado en {csv_path}")
    return datos


def ajustar_envolvente(j2, j3, ancho_bin=2.5, pct=(1.0, 99.0), n_min=5):
    """Limites de J3 por bin de J2, desde muestras tomadas a mano (grados API).

    Por bin toma los percentiles `pct` de J3 y no el min/max crudo, asi una
    lectura suelta rara no fija un limite. Si solo unas pocas muestras tocaron
    el tope, el percentil queda un poco adentro: el error va para el lado
    seguro (rango mas angosto). Los bins con menos de `n_min` muestras se
    descartan y cuentan como NO medidos.

    Ademas ajusta una recta a lo(J2) y a hi(J2) y guarda el residuo maximo:
    si es chico, el limite es una desigualdad lineal y no hace falta la tabla.
    """
    j2 = np.asarray(j2, dtype=np.float64)
    j3 = np.asarray(j3, dtype=np.float64)
    ok = np.isfinite(j2) & np.isfinite(j3)
    j2, j3 = j2[ok], j3[ok]
    bordes = np.arange(np.floor(j2.min() / ancho_bin) * ancho_bin,
                       j2.max() + ancho_bin, ancho_bin)
    idx = np.digitize(j2, bordes) - 1
    c, lo, hi, n = [], [], [], []
    for b in range(len(bordes) - 1):
        sel = j3[idx == b]
        if sel.size < n_min:
            continue
        c.append(0.5 * (bordes[b] + bordes[b + 1]))
        lo.append(np.percentile(sel, pct[0]))
        hi.append(np.percentile(sel, pct[1]))
        n.append(sel.size)
    env = {"j2": np.array(c), "lo": np.array(lo), "hi": np.array(hi),
           "n": np.array(n), "ancho_bin": float(ancho_bin)}
    if len(c) >= 2:
        for lado in ("lo", "hi"):
            m, b0 = np.polyfit(env["j2"], env[lado], 1)
            res = env[lado] - (m * env["j2"] + b0)
            env[f"recta_{lado}"] = (float(m), float(b0), float(np.abs(res).max()))
    return env


def limites_j3(j2_deg, env, margen_deg=0.0):
    """(lo, hi) de J3 en grados API para un J2 dado (escalar o array).

    Interpola la tabla de `ajustar_envolvente`. Devuelve NaN -- nunca un valor
    extrapolado -- fuera del rango de J2 medido y dentro de un hueco entre bins
    medidos, porque una pose que nadie midio no puede leerse como permitida.
    `margen_deg` achica el intervalo por los dos lados; si lo cierra, NaN.
    """
    x = np.asarray(j2_deg, dtype=np.float64)
    c = env["j2"]
    if len(c) < 2:
        raise ValueError(f"la envolvente tiene {len(c)} bins medidos; hacen falta al menos 2")
    lo = np.interp(x, c, env["lo"]) + margen_deg
    hi = np.interp(x, c, env["hi"]) - margen_deg
    medio = 0.5 * env["ancho_bin"]
    fuera = (x < c[0] - medio) | (x > c[-1] + medio)
    # Hueco: los dos bins medidos que rodean a x estan a mas de un bin de distancia.
    k = np.clip(np.searchsorted(c, x), 1, len(c) - 1)
    hueco = (c[k] - c[k - 1] > 1.5 * env["ancho_bin"]) & (x > c[k - 1]) & (x < c[k])
    malo = fuera | hueco | (lo > hi)
    lo = np.where(malo, np.nan, lo)
    hi = np.where(malo, np.nan, hi)
    if lo.ndim == 0:
        return float(lo), float(hi)
    return lo, hi

In [106]:
# --- Grabacion: brazo liberado, movido a mano --------------------------------
from erp.io.paths import resolve_repo_path

MEDIR = True          # True solo con el brazo conectado (celda de `mc`) y alguien sosteniendolo
DURACION_S = 120.0     # o cortar antes con "Interrupt kernel": lo grabado se guarda igual
RATE_HZ = 15.0
CSV_ENVOLVENTE = resolve_repo_path("data", "raw", "j2j3_envelope_raw.csv", must_exist=False)

if MEDIR:
    datos_env = grabar_envolvente(mc, DURACION_S, RATE_HZ, CSV_ENVOLVENTE)
else:
    print("MEDIR = False: no se toca el brazo. La celda siguiente ajusta los CSV ya grabados.")

Pose actual (grados API): J1=-0.3, J2=-0.8, J3=-0.4, J4=0.3
Lectura OK: 73 muestras, J2/J3 se movieron 11.2 grados.
t=    5 s  J2=  -4.7  J3= -11.7  n=73  bins de J2 sin muestras: ['0.5', '5.5', '10.5', '15.5', '20.5', '25.5', '30.5', '35.5', '40.5', '45.5', '50.5', '55.5', '60.5', '65.5', '70.5', '75.5', '80.5', '85.5', '90.5']
t=   10 s  J2=  11.7  J3= -30.1  n=149  bins de J2 sin muestras: ['15.5', '20.5', '25.5', '30.5', '35.5', '40.5', '45.5', '50.5', '55.5', '60.5', '65.5', '70.5', '75.5', '80.5', '85.5', '90.5']
t=   15 s  J2=  24.9  J3= -41.8  n=225  bins de J2 sin muestras: ['30.5', '35.5', '40.5', '45.5', '50.5', '55.5', '60.5', '65.5', '70.5', '75.5', '80.5', '85.5', '90.5']
t=   20 s  J2=  36.8  J3= -55.5  n=300  bins de J2 sin muestras: ['45.5', '50.5', '55.5', '60.5', '65.5', '70.5', '75.5', '80.5', '85.5', '90.5']
t=   25 s  J2=  45.7  J3= -63.5  n=375  bins de J2 sin muestras: ['50.5', '55.5', '60.5', '65.5', '70.5', '75.5', '80.5', '85.5', '90.5']
t=   30 s  J2=  55.2 

In [107]:
# --- Ajuste y grafico: corre sin el brazo, desde los CSV grabados ------------
try:
    import matplotlib.pyplot as plt
except ImportError:   # el env `ekf` de esta maquina no lo trae: pip install -e ".[viz]"
    plt = None

archivos = sorted(CSV_ENVOLVENTE.parent.glob("j2j3_envelope_raw*.csv"))
if not archivos:
    print(f"No hay grabaciones en {CSV_ENVOLVENTE.parent}; primero MEDIR = True.")
else:
    grab = {f.name: np.loadtxt(f, delimiter=",", skiprows=1).reshape(-1, 5) for f in archivos}
    todo = np.vstack(list(grab.values()))
    env = ajustar_envolvente(todo[:, 2], todo[:, 3])

    print(f"{len(archivos)} grabacion(es), {len(todo)} muestras, {len(env['j2'])} bins medidos")
    print(f"{'J2':>7} {'J3 lo':>8} {'J3 hi':>8} {'n':>6}")
    for c, lo, hi, n in zip(env["j2"], env["lo"], env["hi"], env["n"]):
        print(f"{c:7.2f} {lo:8.2f} {hi:8.2f} {n:6d}")
    if "recta_lo" in env:
        for lado in ("lo", "hi"):
            m, b0, r = env[f"recta_{lado}"]
            print(f"recta J3_{lado} = {m:+.3f}*J2 {b0:+.2f}   residuo max {r:.2f} grados")

    # Repetibilidad: con mas de una grabacion, la diferencia entre ellas es el margen.
    if len(grab) > 1:
        x = env["j2"]
        por = [limites_j3(x, ajustar_envolvente(d[:, 2], d[:, 3])) for d in grab.values()]
        lo_s = np.nanmax([p[0] for p in por], 0) - np.nanmin([p[0] for p in por], 0)
        hi_s = np.nanmax([p[1] for p in por], 0) - np.nanmin([p[1] for p in por], 0)
        print(f"dispersion entre grabaciones: lo {np.nanmax(lo_s):.2f}, hi {np.nanmax(hi_s):.2f} grados (max)")

    if plt is None:
        print("matplotlib no esta instalado: sin grafico. La tabla y limites_j3 valen igual.")
    else:
        fig, ax = plt.subplots(figsize=(8, 6))
        for nombre, d in grab.items():
            ax.plot(d[:, 2], d[:, 3], ".", ms=2, alpha=0.3, label=nombre)
        xs = np.linspace(env["j2"][0], env["j2"][-1], 400)
        lo_x, hi_x = limites_j3(xs, env)
        ax.plot(xs, lo_x, "k-", lw=2, label="limites_j3: lo")
        ax.plot(xs, hi_x, "k--", lw=2, label="limites_j3: hi")
        # Cajas fijas que el repo usa hoy (JointMap.api_limits_deg y los range del XML).
        ax.add_patch(plt.Rectangle((-2, -92), 92, 152, fill=False, ec="tab:red", lw=1.5,
                                   label="API: J2 -2..90, J3 -92..60"))
        ax.add_patch(plt.Rectangle((0, 0), 90, np.rad2deg(1.05), fill=False, ec="tab:blue", lw=1.5,
                                   ls=":", label="XML: link1 0..90, link2 0..60.2"))
        ax.set_xlabel("J2 [grados API]")
        ax.set_ylabel("J3 [grados API]")
        ax.set_title("Limites de J3 en funcion de J2 (medidos a mano, servos liberados)")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8, loc="best")
        plt.show()

2 grabacion(es), 2004 muestras, 21 bins medidos
     J2    J3 lo    J3 hi      n
  -6.25   -12.30    58.11    379
  -3.75   -12.48    57.30     27
  11.25   -30.14     0.40     67
  13.75   -30.65    54.25      6
  23.75   -42.09    30.07     48
  26.25   -42.54    30.35     28
  36.25   -55.63    -5.45     58
  38.75   -58.72    44.55     12
  43.75   -63.28    57.04     41
  46.25   -63.67    57.04     61
  48.75   -65.05    56.89     43
  53.75   -72.23    42.27      8
  56.25   -74.82   -72.21     70
  61.25   -81.82   -81.12     80
  63.75   -81.82    36.48      6
  66.25   -82.49    36.96      6
  73.75   -92.01    33.45    116
  81.25  -100.81   -58.83     68
  83.75  -101.30    15.43      7
  91.25  -111.09    14.64     59
  93.75  -111.09    13.53    754
recta J3_lo = -1.009*J2 -17.77   residuo max 2.24 grados
recta J3_hi = -0.543*J2 +46.03   residuo max 93.86 grados


ValueError: la envolvente tiene 1 bins medidos; hacen falta al menos 2